In [4]:
import numpy as np
import pandas as pd

In [5]:
class LinearRegression():
    def __init__(self, features, targets, GD = True, iters = 1000, lr = 0.01, batch_size = 32):
        self.iters = iters if GD else None
        self.lr = lr if GD else None
        self.GD = GD
        self.batch_size = batch_size # batch size for mini-batch GD

        self.features = features.astype(float).copy() # if features are int durning normalization it will be truncted to int
        self.targets = targets.reshape(-1,1) # -1 says that no. of rows based on no. of elements, 1 says that only one column

        self.normRange = self.features.max(axis=0)-self.features.min(axis=0)
        self.normMean = self.features.mean(axis=0) # axis=0 says to operate down the rows, giving one element per column
        self.normRange[self.normRange == 0] = 1 # uses boolean indexing to make all 0s in normRange to 1s
        self.normalized = False

        self.bias = 0
        self.weights = np.zeros(self.features.shape[1], 1) # this creates a column vector of zeros with one row for each feature
    

    def normalize(self, features = None):
        '''each column in features is a new feature\n
         for each feature column:\n
         -> subtract with the mean of the column (mean normalization)\n
         -> divide by the range of the column (scaling) '''
        if features is None:
            features = self.features
        features -= self.normMean
        features /= self.normRange

    def train(self):
        if self.normalized is False:
            self.normalize()
            self.normalized = True

        costHistory = []
        if self.GD:
            nSamples = len(self.features)
            nBatches = int(np.ceil(nSamples/self.batch_size))

            n = max(1, self.iters // 10) # self.iters // 10 is floor division by 10, i.e. it returns the integer quotient after division
            for i in range(n):

                # shuffle data for each epoch/iter
                idxs = np.random.permutation(nSamples)
                shuffledX = self.features[idxs]
                shuffledY = self.features[idxs]

                epoch_costs = []
                for j in range[nBatches]:
                    start = j * self.batch_size
                    end = min(nSamples, (j+1)*self.batch_size)
                    X_batch = shuffledX[start:end]
                    Y_batch = shuffledY[start:end]

                    self.updateWeights(X_batch, Y_batch)
                    epoch_costs.append(self.cost_fn(X_batch, Y_batch))

                costHistory.append(np.mean(epoch_costs))